# Construction de notre dataset

## Connexion

In [1]:
import ee
ee.Authenticate(force=True)
ee.Initialize(project='stress-hydrique-491820')
print("✅ Connexion GEE réussie !")

✅ Connexion GEE réussie !


## Définition des points d'échantillonnage

In [2]:
points = {
    'Mandoul':           ee.Geometry.Point([17.4,  8.6]),
    'Moyen_Chari':       ee.Geometry.Point([18.2,  9.1]),
    'Logone_Occidental': ee.Geometry.Point([15.8,  8.9]),
    'Salamat':           ee.Geometry.Point([20.5, 10.8]),
    'Mayo_Kebbi':        ee.Geometry.Point([15.2, 10.1]),
}

print("Points définis :", list(points.keys()))

Points définis : ['Mandoul', 'Moyen_Chari', 'Logone_Occidental', 'Salamat', 'Mayo_Kebbi']


## Extraction des indices pour tous les points

In [10]:
import pandas as pd

def masquer_nuages(image):
    masque = image.select('SCL').neq(9)
    return image.updateMask(masque)

dataset = []  # liste qui contiendra toutes nos lignes

annees = [2019, 2020, 2021, 2022, 2023]
mois = ['06', '07', '08', '09']

for nom_region, point in points.items():

    for annee in annees:
      for mois_num in mois:

          date_debut = f'{annee}-{mois_num}-01'
          date_fin   = f'{annee}-{mois_num}-28'

          collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
              .filterBounds(point) \
              .filterDate(date_debut, date_fin) \
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50)) \
              .map(masquer_nuages) \
              .limit(2)  # max 2 images par mois

          images = collection.toList(5)
          n = collection.size().getInfo()

          for i in range(n):
              image = ee.Image(images.get(i))

              # Calcul des 3 indices
              ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
              ndwi = image.normalizedDifference(['B8', 'B11']).rename('NDWI')
              msi  = image.select('B11').divide(image.select('B8')).rename('MSI')

              # Extraction des valeurs
              valeurs = ndvi.addBands(ndwi).addBands(msi).reduceRegion(
                  reducer=ee.Reducer.mean(),
                  geometry=point,
                  scale=20
              ).getInfo()

              # Date de l'image
              date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd').getInfo()

              # On ajoute une ligne au dataset
              dataset.append({
                  'region':   nom_region,
                  'date':     date,
                  'NDVI':     valeurs.get('NDVI'),
                  'NDWI':     valeurs.get('NDWI'),
                  'MSI':      valeurs.get('MSI'),
              })

# Conversion en DataFrame pandas
df = pd.DataFrame(dataset)
print(df)
print("\nNombre de lignes :", len(df))

         region        date      NDVI      NDWI       MSI
0       Mandoul  2019-06-05  0.335336 -0.257854  1.694888
1       Mandoul  2019-06-10  0.389269 -0.185882  1.456645
2       Mandoul  2019-07-25  0.730159  0.205631  0.658882
3       Mandoul  2019-08-09  0.840020  0.335108  0.498006
4       Mandoul  2019-08-14  0.840686  0.389776  0.439081
..          ...         ...       ...       ...       ...
161  Mayo_Kebbi  2023-07-17  0.394632 -0.089163  1.195783
162  Mayo_Kebbi  2023-08-11  0.353871  0.451543  0.377844
163  Mayo_Kebbi  2023-08-26  0.162397 -0.028660  1.059012
164  Mayo_Kebbi  2023-09-10  0.495628 -0.011927  1.024142
165  Mayo_Kebbi  2023-09-20  0.395013  0.216898  0.643523

[166 rows x 5 columns]

Nombre de lignes : 166


## Ajout des labels

In [11]:
# Règles de classification combinées
# Basées sur littérature télédétection Sahel

def classifier_stress(row):
    ndvi = row['NDVI']
    ndwi = row['NDWI']
    msi  = row['MSI']

    if ndvi > 0.5 and ndwi > 0.2 and msi < 0.6:
        return 'Normal'
    elif ndvi > 0.3 and ndwi > 0.0 and msi < 0.9:
        return 'Stress_leger'
    elif ndvi > 0.1 and msi < 1.3:
        return 'Stress_modere'
    else:
        return 'Stress_severe'

df['label'] = df.apply(classifier_stress, axis=1)

print(df[['region', 'date', 'NDVI', 'NDWI', 'MSI', 'label']])
print("\nDistribution des classes :")
print(df['label'].value_counts())

         region        date      NDVI      NDWI       MSI          label
0       Mandoul  2019-06-05  0.335336 -0.257854  1.694888  Stress_severe
1       Mandoul  2019-06-10  0.389269 -0.185882  1.456645  Stress_severe
2       Mandoul  2019-07-25  0.730159  0.205631  0.658882   Stress_leger
3       Mandoul  2019-08-09  0.840020  0.335108  0.498006         Normal
4       Mandoul  2019-08-14  0.840686  0.389776  0.439081         Normal
..          ...         ...       ...       ...       ...            ...
161  Mayo_Kebbi  2023-07-17  0.394632 -0.089163  1.195783  Stress_modere
162  Mayo_Kebbi  2023-08-11  0.353871  0.451543  0.377844   Stress_leger
163  Mayo_Kebbi  2023-08-26  0.162397 -0.028660  1.059012  Stress_modere
164  Mayo_Kebbi  2023-09-10  0.495628 -0.011927  1.024142  Stress_modere
165  Mayo_Kebbi  2023-09-20  0.395013  0.216898  0.643523   Stress_leger

[166 rows x 6 columns]

Distribution des classes :
label
Stress_modere    52
Stress_severe    47
Stress_leger     41
Normal

### Sauvegarde du dataset  

In [12]:
df.to_csv('dataset_stress_hydrique.csv', index=False)
print("✅ Dataset sauvegardé !")

✅ Dataset sauvegardé !
